In [10]:
import pandas as pd
import numpy as np
import os
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [11]:
def Read_data():
    df = pd.read_csv(f'{os.getcwd()}/data/ECERIECS.csv')
    print(f'Tamaño del dataset original: {df.shape}')
    estados_df = pd.read_csv(f'{os.getcwd()}/data/CentrosDeCostoEstado.csv')
    dict2 = dict(zip(estados_df['CENTRO'], estados_df['ESTADO']))
    centros = pd.read_csv(f'{os.getcwd()}/data/CentrosDeCosto.csv')
    dict_centros = dict(zip(centros['CentroCostoClave'], centros['CentroCostoId']))
    return df, dict2, dict_centros

def recod_columns(df):
    labels = [
        'Tabaco',
        'Alcohol',
        'Marihuana',
        'Hachis',
        'Cocaina',
        'Crack',
        'Otras Presentaciones (Basuco o pasta base, cocaina negra)',
        'Solventes y removedores',
        'Pegamento',
        'Esmaltes y pinturas',
        'Otros (aire comprimido, gasolinas y combustibles)',
        'Anfetaminas',
        'Metanfetaminas',
        'MDMA(extasis) y metanfetaminas alucinogenas (DMT)',
        'Otros (derivados anfetaminicos)',
        'LSD',
        'Plantas alucinogenas y derivados',
        'Otras (PCP, ketamina, excepto metanfetamina)',
        'Benzodiazepinas',
        'Rohypnol',
        'Otras (sedantes hiptnoticos, GHB)',
        'Heroina',
        'Opiaceos sinteticos (propoxifeno, nailbufina)',
        'Opio y opiodes (morfina, codeina)',
        'Con utilidad medica (Prozac, Paxil, Carbamazepina)',
        'Otras Sustancias'
    ]

    mapping = {}
    for i, label in enumerate(labels, start=1):
        mapping[f'abst_{i}']    = f'Abstinencia{label}'
        mapping[f'ocasion_{i}'] = f'Ocasion{label}'
        mapping[f'av_{i}']     = f'{label}'
        mapping[f'edad_inicio_{i}'] = f'EdadInicio{label}'

    return df.rename(columns=mapping)

def dict_impacto():
    return {
        1 : 'Tabaco',
        2 : 'Alcohol',
        3 : 'Marihuana',
        4 : 'Hachis',
        5 : 'Cocaina',
        6 : 'Crack',
        7 : 'Otras Presentaciones (Basuco o pasta base, cocaina negra)',
        8 : 'Solventes y removedores',
        9 : 'Pegamento',
        10 : 'Esmaltes y pinturas',
        11 : 'Otros (aire comprimido, gasolinas y combustibles)',
        12 : 'Anfetaminas',
        13 : 'Metanfetaminas',
        14 : 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)',
        15 : 'Otros (derivados anfetaminicos)',
        16 : 'LSD',
        17 : 'Plantas alucinogenas y derivados',
        18 : 'Otras (PCP, ketamina, excepto metanfetamina)',
        19 : 'Benzodiazepinas',
        20 : 'Rohypnol',
        21 : 'Otras (sedantes hiptnoticos, GHB)',
        22 : 'Heroina',
        23 : 'Opiaceos sinteticos (propoxifeno, nailbufina)',
        24 : 'Opio y opiodes (morfina, codeina)',
        25 : 'Con utilidad medica (Prozac, Paxil, Carbamazepina)',
        26 : 'Otras Sustancias'
    }

def rangos_edades(edad):
    if pd.isna(edad):
        return np.nan
    elif edad < 10: 
        return '0-9'
    elif edad < 15:
        return '10-14'
    elif edad < 20:
        return '15-19'
    elif edad < 25:
        return '20-24'
    elif edad < 30:
        return '25-29'
    elif edad < 35:
        return '30-34'
    elif edad < 40:
        return '35-39'
    elif edad < 45:
        return '40-44'
    elif edad >= 45:
        return '45+'
    
def Recod_sexo(x):
    if x == '1. Hombre':
        return 'Hombre'
    elif x == '2. Mujer':
        return 'Mujer'
    
def Recod_mig(x):
    if x == 'No' or pd.isna(x):
        return 0
    else:
        return 1
    
def diccionario_escolaridad():
    return {
        ('1. Sin estudios (no sabe leer ni escribir)1. Estudios concluidos'): 'Sin Estudios',
        ('1. Sin estudios (no sabe leer ni escribir)2. Estudios no concluidos'): 'Sin Estudios',
        ('1. Sin estudios (no sabe leer ni escribir)3. En curso'): 'Sin Estudios',
        
        ('2. Sin estudios (sabe leer y escribir)1. Estudios concluidos'): 'Sin Estudios',
        ('2. Sin estudios (sabe leer y escribir)2. Estudios no concluidos'): 'Sin Estudios',
        ('2. Sin estudios (sabe leer y escribir)3. En curso'): 'Sin Estudios',
        
        ('3. Primaria1. Estudios concluidos'): 'Primaria',
        ('3. Primaria2. Estudios no concluidos'): 'Sin Estudios',
        ('3. Primaria3. En curso'): 'Sin Estudios',
        
        ('4. Secundaria1. Estudios concluidos'): 'Secundaria',
        ('4. Secundaria2. Estudios no concluidos'): 'Primaria',
        ('4. Secundaria3. En curso'): 'Primaria',
        
        ('5. Estudios técnicos o comerciales1. Estudios concluidos'): 'Preparatoria o Carrera Técnica',
        ('5. Estudios técnicos o comerciales2. Estudios no concluidos'): 'Secundaria',
        ('5. Estudios técnicos o comerciales3. En curso'): 'Secundaria',
        
        ('6. Bachillerato o bachillerato técnico1. Estudios concluidos'): 'Preparatoria o Carrera Técnica',
        ('6. Bachillerato o bachillerato técnico2. Estudios no concluidos'): 'Secundaria',
        ('6. Bachillerato o bachillerato técnico3. En curso'): 'Secundaria',
        
        ('7. Estudios superiores1. Estudios concluidos'): 'Estudios Superiores',
        ('7. Estudios superiores2. Estudios no concluidos'): 'Preparatoria o Carrera Técnica',
        ('7. Estudios superiores3. En curso'): 'Preparatoria o Carrera Técnica',
        
        ('8. Estudios de posgrado1. Estudios concluidos'): 'Estudios de Posgrado',
        ('8. Estudios de posgrado2. Estudios no concluidos'): 'Estudios Superiores',
        ('8. Estudios de posgrado3. En curso'): 'Estudios Superiores',
        }

def diccionario_ocupacion():
    return {
        '1. Estudiante de tiempo completo': 'Estudiante',
        '2. Estudiante con actividad laboral': 'Estudiante',
        '3. Con actividad laboral estable (más de 6 meses)': 'Con actividad laboral',
        '4. Con actividad laboral reciente o inestable (menos de 6 meses)': 'Con actividad laboral',
        '5. Desempleado(a) (buscó empleo en el último mes)': 'Sin ocupación',
        '6. Desocupado(a) (no buscó empleo en el último mes)': 'Sin ocupación',
        '7. Hogar': 'Hogar',
        '8. Pensionado(a) o jubilado(a)': 'Pensionado o jubilado',
        '8. Pensionado(a) o jubilada(a)': 'Pensionado o jubilado',
        '9. Personal del sector salud': 'Con actividad laboral'
        }

def diccionario_estrato():
    return {
        '1. Alto': 'Alto',
        '2. Medio alto': 'Medio Alto',
        '3. Medio': 'Medio Alto',
        '4. Medio bajo': 'Medio Bajo',
        '5. Bajo': 'Bajo',
        '6. Pobreza extrema o marginación': 'Muy Bajo'
        }

def Mod_data(df, estados, dict_centros):
    dict3 = {'CIUDAD DE MÉXICO':"CDMX", 'ESTADO DE MÉXICO': "EMEX", 'JALISCO':"JAL", 'SINALOA': 'SIN', 'BAJA CALIFORNIA':"BC", 'CHIHUAHUA': "CHH", 'GUANAJUATO': 'GTO', 'QUINTANA ROO':"QNTROO", 'COAHUILA':  "COAH", 'NUEVO LEÓN':  "NVL", 'MICHOACÁN': "MICH", 'GUERRERO': "GRO", 'COLIMA': "COL", 'BAJA CALIFORNIA SUR':"BCS", 'TAMAULIPAS': "TAM",
        'VERACRUZ': "VRC", 'SONORA': "SON", 'PUEBLA': "PBL", 'DURANGO': "DUR", 'AGUASCALIENTES':  "AGS", 'YUCATÁN':"YCT", 'HIDALGO':"HDO",'ZACATECAS':"ZAC", 'QUERÉTARO': "QRO", 'SAN LUIS POTOSÍ': "SLP" , 'OAXACA': "OAX", 'TABASCO': "TBS", 'CHIAPAS': "CHPS", 'MORELOS':"MOR", 'TLAXCALA': "TLX", 'CAMPECHE':"CAM", 'NAYARIT':"NAY"}
    df['FolioId'] = df['folio']
    df['CentroCostoId'] = df['cec'].map(dict_centros)
    print (f'Tamaño del dataset después de filtrar por CentroCostoId: {df.shape}')
    df = df[(df['CentroCostoId'] > 58) |(df['CentroCostoId'].isin([48, 49]))]
    print (f'Tamaño del dataset después de filtrar por CentroCostoId: {df.shape}')
    df['Estado'] = df['cec'].map(estados).map(dict3)
    df['Edad'] = df['edad'].apply(rangos_edades)
    df['Edad_Años'] = df['edad']
    df['SexoId'] = df['sexo'].apply(Recod_sexo)
    df['Migracion'] = df['migrado_pais'].apply(Recod_mig)
    df['ComunEstadoCivilId'] = df['estado_civil'].astype(str).str.replace(r'^\d+\.\s*', '', regex=True)
    df['EscolaridadConcat'] = df['escolaridad1'] + df['escolaridad2']
    df['ComunEscolaridadId'] = df['EscolaridadConcat'].map(diccionario_escolaridad()).fillna(df['EscolaridadConcat'])
    df['ComunOcupacionId'] = df['prin_ocupacion_opcion'].map(diccionario_ocupacion())
    df['ComunEstratoSocialId'] = df['estrato_social'].map(diccionario_estrato())
    df['PerteneceComunidadLGBTTTI'] = 0
    df['PerteneceComunidadIndigena'] = 0
    df['PoblacionAfromexicanaAfroamericana'] = 0
    df['DiscapacidadPerceptual'] = 0
    df['fecha_captura_1'] = pd.to_datetime(df['fecha_captura_1'])
    df['Año'] = df['fecha_captura_1'].dt.year
    df['Mes'] = df['fecha_captura_1'].dt.month
    df['Semestre'] = df['Mes'].apply(lambda x: 1 if x in range(1, 7) else 2)
    df['Mes'] = df['Año'].astype(str) + '-' + df['Mes'].astype(str).str.zfill(2)
    df['Semestre'] = df['Año'].astype(str) + '-' + df['Semestre'].astype(str).str.zfill(2)
    df['DrogaImpacto'] = df['drog_may_imp_12_mes']
    df['DrogaImpacto'] = df['DrogaImpacto'].map(dict_impacto())
    print (f'Tamaño del dataset después de filtrar por Año: {df.shape}')
    df = df[df['Año'] != 2021]
    print (f'Tamaño del dataset después de filtrar por Año: {df.shape}')
    df.drop(columns=['estado_civil','EscolaridadConcat','escolaridad1', 'escolaridad2', 'estrato_social', 'prin_ocupacion_opcion', 'migrado_pais', 'folio', 'cec', 'sexo', 'edad', 'fecha1','fecha_captura_1', 'drog_may_imp_12_mes'], inplace=True)
    return df

def OrdenCols (df):
    cols = ['FolioId','Edad','SexoId','Migracion','ComunEstadoCivilId','ComunEscolaridadId','ComunOcupacionId','ComunEstratoSocialId','PerteneceComunidadLGBTTTI','PerteneceComunidadIndigena','PoblacionAfromexicanaAfroamericana','DiscapacidadPerceptual']
    cols2 =[]
    for col in df.columns:
        if col not in cols:
            cols2.append(col)
    return df[cols + cols2]

codigo_a_motivo = {
    1: 'ConsumoDeDrogas',
    2: 'ConsumoDeBebidasAlcoholicas',
    3: 'ConsumoDeTabaco',
    4: 'Ludopatia',
    5: 'Otro',
    6: 'ConsumoDeDrogas', #ConsumoDeMarihuana
    7: 'TrastornosMentales', #SaludMental 
    8: 'Depresion',
    9: 'Psicosis',
    10: 'Epilepsia',
    11: 'TrastornosMentales',
    12: 'Demencia',
    13: 'Autolesion',
    14: 'Suicidio',
    15: 'Ansiedad',
    16: 'Otro'
}

codigo_a_motivo_2 = {
    1: 'ProblemasSalud',
    3: 'ProblemasFamiliares',
    2: 'AccidentesAsociados',
    4: 'ProblemasEscolares',
    5: 'ProblemasLaborales',
    6: 'ProblemasPsicologicos',
    7: 'ProblemasLegales',
    8: 'ConductaAntisocial',
    9: 'ProblemasOtros'
    }

def MotivoConsulta(df):
    # 1) Extraigo UNA vez la lista de códigos enteros
    df['codigos'] = (
        df['mot_exp_cond_opcion']
        .astype(str)
        .str.findall(r'\d+')           # lista de strings
        .apply(lambda lst: [int(x) for x in lst])
    )

    # 2) Agrupo los códigos por nombre de motivo
    name_to_codes = defaultdict(list)
    for codigo, nombre in codigo_a_motivo.items():
        name_to_codes[nombre].append(codigo)

    # 3) Creo una columna por cada motivo, comprobando si alguno de sus códigos está en la lista
    for nombre, codigos_asociados in name_to_codes.items():
        df[nombre] = df['codigos'].apply(
            lambda lst: int(any(c in lst for c in codigos_asociados))
        )

    # 4) Opcional: elimino columnas intermedias
    return df.drop(columns=['mot_exp_cond_opcion', 'codigos'])

def ProblemasAsociados(df):
    # 1) Extraigo UNA sola vez la lista de enteros
    df['codigos'] = (
        df['prob_alt_asoc_cons_sust']
        .astype(str)
        .str.findall(r'\d+')              # lista de strings digito+
        .apply(lambda lst: [int(x) for x in lst])
    )

    # 2) Creo cada dummy iterando sobre tu mapeo
    for codigo, nombre in codigo_a_motivo_2.items():
        df[nombre] = df['codigos'].apply(lambda lst: int(codigo in lst))

    # 3) Elimino sólo lo que ya no sirve
    return df.drop(columns=['prob_alt_asoc_cons_sust', 'codigos'])

def UltimoMes(df):
    # 1) identifico columnas de Ocasión
    ocasion_cols = [c for c in df.columns if c.startswith('Ocasion')]
    
    # 2) para cada sustancia, creo directamente la dummy vectorizada
    for oc_col in ocasion_cols:
        ab_col = oc_col.replace('Ocasion', 'Abstinencia')
        sust = oc_col.replace('Ocasion', '')
        
        # definiendo la máscara: 1–3 en Ocasión y 1 en Abstinencia
        mask = df[oc_col].between(1, 3) & (df[ab_col] == 1)
        
        # asigno la columna nueva de una sola vez
        df[sust + 'UltimoMes'] = mask.astype(int)
    
    # 3) eliminamos las columnas intermedias
    cols_to_drop = ocasion_cols + [c.replace('Ocasion', 'Abstinencia') for c in ocasion_cols]
    return df.drop(columns=cols_to_drop)

def Denominadores(df):
    sustancias_legales = ['Tabaco', 'Alcohol']
    sustancias_ilegales = ['Marihuana', 'Hachis', 'Cocaina', 'Crack', 
                        'Otras Presentaciones (Basuco o pasta base, cocaina negra)', 
                        'Solventes y removedores', 'Pegamento', 
                        'Esmaltes y pinturas', 'Otros (aire comprimido, gasolinas y combustibles)', 
                        'Anfetaminas', 'Metanfetaminas', 
                        'MDMA(extasis) y metanfetaminas alucinogenas (DMT)', 
                        'Otros (derivados anfetaminicos)', 'LSD', 
                        'Plantas alucinogenas y derivados', 
                        'Otras (PCP, ketamina, excepto metanfetamina)', 
                        'Benzodiazepinas', 'Rohypnol', 
                        'Otras (sedantes hiptnoticos, GHB)', 
                        'Heroina', 'Opiaceos sinteticos (propoxifeno, nailbufina)', 
                        'Opio y opiodes (morfina, codeina)', 
                        'Con utilidad medica (Prozac, Paxil, Carbamazepina)', 
                        'Otras Sustancias']

    df_legales = df[df['DrogaImpacto'].isin(sustancias_legales)]

    # 2. Detectar filas que tienen al menos un valor distinto de 0 en sustancias ilegales
    violaciones = df_legales[sustancias_ilegales].ne(0).any(axis=1)

    # 3. Ver filas que incumplen
    df_sinconsumoilegales = df_legales[~violaciones]
    
    df_EceRiegsIlegales = df[~df['FolioId'].isin(df_sinconsumoilegales['FolioId'])]
    print(f'Tamaño del dataset después de filtrar por consumo de drogas ilegales: {df_EceRiegsIlegales.shape}')
    df_EceRiegsIlegales.to_csv('result/ECERIECSIlegales(EdadInicio).csv', index=False)
    df.to_csv('result/ECERIECSLegalesIlegales_mod(EdadInicio).csv', index=False)   
    print (len(df))

def dict_impactosust():
    return {
        1 : 'Tabaco',
        2 : 'Alcohol',
        3 : 'Marihuana',
        4 : 'Hachis',
        5 : 'Cocaina',
        6 : 'Crack',
        7 : 'Otras Presentaciones (Basuco o pasta base, cocaina negra)',
        8 : 'Solventes y removedores',
        9 : 'Pegamento',
        10 : 'Esmaltes y pinturas',
        11 : 'Otros (aire comprimido, gasolinas y combustibles)',
        12 : 'Anfetaminas',
        13 : 'Metanfetaminas',
        14 : 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)',
        15 : 'Otros (derivados anfetaminicos)',
        16 : 'LSD',
        17 : 'Plantas alucinogenas y derivados',
        18 : 'Otras (PCP, ketamina, excepto metanfetamina)',
        19 : 'Benzodiazepinas',
        20 : 'Rohypnol',
        21 : 'Otras (sedantes hiptnoticos, GHB)',
        22 : 'Heroina',
        23 : 'Opiaceos sinteticos (propoxifeno, nailbufina)',
        24 : 'Opio y opiodes (morfina, codeina)',
        25 : 'Con utilidad medica (Prozac, Paxil, Carbamazepina)',
        26 : 'Otras Sustancias'
    }

def Grupos(df):
    df['Marihuana'] = df[['Marihuana', 'Hachis']].max(axis=1)
    df['Cocaína'] = df[['Cocaina', 'Crack', 'Otras Presentaciones (Basuco o pasta base, cocaina negra)']].max(axis=1)
    df['Inhalables'] = df[['Solventes y removedores', 'Pegamento', 'Esmaltes y pinturas', 'Otros (aire comprimido, gasolinas y combustibles)']].max(axis=1)
    df['Metanfetaminas'] = df[['Anfetaminas', 'Metanfetaminas', 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)', 'Otros (derivados anfetaminicos)']].max(axis=1)
    df['Alucinógenos'] = df[['LSD', 'Plantas alucinogenas y derivados', 'Otras (PCP, ketamina, excepto metanfetamina)']].max(axis=1)
    df['Medicamentos'] = df[['Benzodiazepinas', 'Rohypnol', 'Otras (sedantes hiptnoticos, GHB)', 'Con utilidad medica (Prozac, Paxil, Carbamazepina)']].max(axis=1)
    df['Opioides'] = df[['Heroina', 'Opiaceos sinteticos (propoxifeno, nailbufina)', 'Opio y opiodes (morfina, codeina)']].max(axis=1)
    cols_a_eliminar = [col for col in dict_impactosust().values() if col not in ['Tabaco', 'Alcohol', 'Metanfetaminas', 'Otras Sustancias', 'Marihuana']]
    df.drop(columns=cols_a_eliminar, inplace=True)
    return df

def UltimoMesGrupos(df):
    df['TabacoUM'] = df['TabacoUltimoMes']
    df['AlcoholUM'] = df['AlcoholUltimoMes']
    df['MarihuanaUM'] = df[['MarihuanaUltimoMes', 'HachisUltimoMes']].max(axis=1)
    df['CocaínaUM'] = df[['CocainaUltimoMes', 'CrackUltimoMes', 'Otras Presentaciones (Basuco o pasta base, cocaina negra)UltimoMes']].max(axis=1)
    df['InhalablesUM'] = df[['Solventes y removedoresUltimoMes', 'PegamentoUltimoMes', 'Esmaltes y pinturasUltimoMes', 'Otros (aire comprimido, gasolinas y combustibles)UltimoMes']].max(axis=1)
    df['MetanfetaminasUM'] = df[['AnfetaminasUltimoMes', 'MetanfetaminasUltimoMes', 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)UltimoMes', 'Otros (derivados anfetaminicos)UltimoMes']].max(axis=1)
    df['AlucinógenosUM'] = df[['LSDUltimoMes', 'Plantas alucinogenas y derivadosUltimoMes', 'Otras (PCP, ketamina, excepto metanfetamina)UltimoMes']].max(axis=1)
    df['MedicamentosUM'] = df[['BenzodiazepinasUltimoMes', 'RohypnolUltimoMes', 'Otras (sedantes hiptnoticos, GHB)UltimoMes', 'Con utilidad medica (Prozac, Paxil, Carbamazepina)UltimoMes']].max(axis=1)
    df['OpioidesUM'] = df[['HeroinaUltimoMes', 'Opiaceos sinteticos (propoxifeno, nailbufina)UltimoMes', 'Opio y opiodes (morfina, codeina)UltimoMes']].max(axis=1)
    df['Otras SustanciasUM'] = df[['Otras SustanciasUltimoMes']].max(axis=1)
    df['UltimoMes'] = df[['TabacoUM', 'AlcoholUM', 'MarihuanaUM', 'CocaínaUM', 'InhalablesUM', 'MetanfetaminasUM', 'AlucinógenosUM', 'MedicamentosUM', 'OpioidesUM', 'Otras SustanciasUM']].max(axis=1)
    colEliminar = [col for col in df.columns if col.endswith('UltimoMes')]
    df.drop(columns=colEliminar , inplace=True) 
    return df

def dict_grupos():
    return {
        'Tabaco' : 'Tabaco',
        'Alcohol' : 'Alcohol',
        'Marihuana' : 'Marihuana',
        'Hachis' : 'Marihuana',
        'Cocaina' : 'Cocaína',
        'Crack' : 'Cocaína',
        'Otras Presentaciones (Basuco o pasta base, cocaina negra)' : 'Cocaína',
        'Solventes y removedores' : 'Inhalables',
        'Pegamento' : 'Inhalables',
        'Esmaltes y pinturas' : 'Inhalables',
        'Otros (aire comprimido, gasolinas y combustibles)' : 'Inhalables',
        'Anfetaminas': 'Metanfetaminas',
        'Metanfetaminas' : 'Metanfetaminas',
        'MDMA(extasis) y metanfetaminas alucinogenas (DMT)' : 'Metanfetaminas',
        'Otros (derivados anfetaminicos)' : 'Metanfetaminas',
        'LSD' : 'Alucinógenos',
        'Plantas alucinogenas y derivados' : 'Alucinógenos',
        'Otras (PCP, ketamina, excepto metanfetamina)' : 'Alucinógenos',
        'Benzodiazepinas' : 'Medicamentos',
        'Rohypnol' : 'Medicamentos',
        'Otras (sedantes hiptnoticos, GHB)' : 'Medicamentos',
        'Heroina' : 'Opioides',
        'Opiaceos sinteticos (propoxifeno, nailbufina)' : 'Opioides',
        'Opio y opiodes (morfina, codeina)' : 'Opioides',
        'Con utilidad medica (Prozac, Paxil, Carbamazepina)' : 'Medicamentos',
        'Otras Sustancias' : 'Otras Sustancias'
    }

def edades_inicio(df):
    df['EdadInicioTabaco'] = df['EdadInicioTabaco']
    df['EdadInicioAlcohol'] = df['EdadInicioAlcohol']
    df['EdadInicioMarihuana'] = df[['EdadInicioMarihuana', 'EdadInicioHachis']].max(axis=1)
    df['EdadInicioCocaína'] = df[['EdadInicioCocaina', 'EdadInicioCrack', 'EdadInicioOtras Presentaciones (Basuco o pasta base, cocaina negra)']].max(axis=1)
    df['EdadInicioInhalables'] = df[['EdadInicioSolventes y removedores', 'EdadInicioPegamento', 'EdadInicioEsmaltes y pinturas', 'EdadInicioOtros (aire comprimido, gasolinas y combustibles)']].max(axis=1)
    df['EdadInicioMetanfetaminas'] = df[['EdadInicioAnfetaminas', 'EdadInicioMetanfetaminas', 'EdadInicioMDMA(extasis) y metanfetaminas alucinogenas (DMT)', 'EdadInicioOtros (derivados anfetaminicos)']].max(axis=1)
    df['EdadInicioAlucinógenos'] = df[['EdadInicioLSD', 'EdadInicioPlantas alucinogenas y derivados', 'EdadInicioOtras (PCP, ketamina, excepto metanfetamina)']].max(axis=1)
    df['EdadInicioMedicamentos'] = df[['EdadInicioBenzodiazepinas', 'EdadInicioRohypnol', 'EdadInicioOtras (sedantes hiptnoticos, GHB)', 'EdadInicioCon utilidad medica (Prozac, Paxil, Carbamazepina)']].max(axis=1)
    df['EdadInicioOpioides'] = df[['EdadInicioHeroina', 'EdadInicioOpiaceos sinteticos (propoxifeno, nailbufina)', 'EdadInicioOpio y opiodes (morfina, codeina)']].max(axis=1)
    df['EdadInicioOtrasSustancias'] = df[['EdadInicioOtras Sustancias']].max(axis=1)
    df['EdadInicioTodasSustancias'] = df[['EdadInicioTabaco', 'EdadInicioAlcohol', 'EdadInicioMarihuana', 'EdadInicioCocaína', 'EdadInicioInhalables', 'EdadInicioMetanfetaminas', 'EdadInicioAlucinógenos', 'EdadInicioMedicamentos', 'EdadInicioOpioides', 'EdadInicioOtras Sustancias']].max(axis=1)
    colEliminar = [col for col in df.columns if col.startswith('EdadInicio') and col not in ['EdadInicioTabaco', 'EdadInicioAlcohol', 'EdadInicioMarihuana', 'EdadInicioCocaína', 'EdadInicioInhalables', 'EdadInicioMetanfetaminas', 'EdadInicioAlucinógenos', 'EdadInicioMedicamentos', 'EdadInicioOpioides', 'EdadInicioOtras Sustancias', 'EdadInicioTodasSustancias']]
    df.drop(columns=colEliminar , inplace=True) 
    return df

In [12]:
def main():
    df_historico, estados, dict_centros = Read_data()
    df_historico_modified = recod_columns(df_historico)
    df_historico_modified = Mod_data(df_historico_modified, estados, dict_centros)
    df_historico_modified = OrdenCols(df_historico_modified)
    df_historico_modified = MotivoConsulta(df_historico_modified)
    df_historico_modified = ProblemasAsociados(df_historico_modified)
    print (f'Tamaño del dataset después de procesar MotivoConsulta y ProblemasAsociados: {df_historico_modified.shape}')
    df_historico_modified = UltimoMes(df_historico_modified)
    print (f'Tamaño del dataset después de procesar MotivoConsulta, ProblemasAsociados y UltimoMes: {df_historico_modified.shape}')
    # list_elim = [col for col in df_historico_modified.columns if col.startswith('edad_inicio')]
    # df_historico_modified.drop(columns=list_elim, inplace=True)
    Denominadores(df_historico_modified)
    return df_historico_modified

In [13]:
if __name__ == "__main__":
    df = main()

Tamaño del dataset original: (314645, 119)
Tamaño del dataset después de filtrar por CentroCostoId: (314645, 121)
Tamaño del dataset después de filtrar por CentroCostoId: (314601, 121)
Tamaño del dataset después de filtrar por Año: (314601, 139)
Tamaño del dataset después de filtrar por Año: (298382, 139)
Tamaño del dataset después de procesar MotivoConsulta y ProblemasAsociados: (298382, 145)
Tamaño del dataset después de procesar MotivoConsulta, ProblemasAsociados y UltimoMes: (298382, 119)
Tamaño del dataset después de filtrar por consumo de drogas ilegales: (250663, 119)
298382


,FolioId,Edad,SexoId,Migracion,ComunEstadoCivilId,ComunEscolaridadId,ComunOcupacionId,ComunEstratoSocialId,PerteneceComunidadLGBTTTI,PerteneceComunidadIndigena,...,AlucinógenosUM,MedicamentosUM,OpioidesUM,Otras SustanciasUM,EdadInicioCocaína,EdadInicioInhalables,EdadInicioAlucinógenos,EdadInicioMedicamentos,EdadInicioOpioides,EdadInicioTodasSustancias
0,765,10-14,Hombre,0,Soltero(a),Primaria,NaN,Medio Bajo,0,0,...,0,0,0,0,NaN,14.0,NaN,NaN,NaN,14.0
1,766,35-39,Hombre,0,Casado(a),NaN,Pensionado o jubilado,Medio Bajo,0,0,...,0,0,0,0,NaN,NaN,NaN,19.0,NaN,28.0
2,132,NaN,Mujer,0,nan,NaN,NaN,NaN,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,769,30-34,Mujer,0,Divorciado(a),Preparatoria o Carrera Técnica,Sin ocupación,Bajo,0,0,...,0,0,0,0,28.0,NaN,NaN,NaN,NaN,28.0
4,770,25-29,Hombre,0,Soltero(a),Secundaria,Sin ocupación,Medio Bajo,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314640,310156,10-14,Hombre,0,Soltero(a),Primaria,Estudiante,Medio Alto,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,12.0
314641,321627,15-19,Hombre,0,Soltero(a),Preparatoria o Carrera Técnica,Sin ocupación,Bajo,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,17.0
314642,310967,30-34,Hombre,0,Unión libre,Secundaria,Estudiante,Bajo,0,0,...,0,0,0,0,29.0,NaN,NaN,NaN,NaN,29.0
314643,310855,35-39,Mujer,0,Soltero(a),Estudios Superiores,Con actividad laboral,Medio Alto,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,27.0


In [14]:
df = Grupos(df.copy())
print (f'Tamaño del dataset Grupo Completo: {df.shape}')

Tamaño del dataset Grupo Completo: (298382, 103)


In [15]:
df = UltimoMesGrupos(df.copy())
print (f'Tamaño del dataset Completo: {df.shape}')

Tamaño del dataset Completo: (298382, 87)


In [16]:
df['DrogaImpacto'] = df['DrogaImpacto'].map(dict_grupos())
print (f'Tamaño del dataset después de agrupar DrogaImpacto: {df.shape}')

Tamaño del dataset después de agrupar DrogaImpacto: (298382, 87)


In [17]:
df = edades_inicio(df.copy())
print (f'Tamaño del dataset después de agrupar EdadInicio: {df.shape}')

Tamaño del dataset después de agrupar EdadInicio: (298382, 72)


In [18]:
df.to_csv('result/GruposLegalesIlegales_ECERIECS_mod(EdadInicio).csv', index=False)